# tVNS sensorimotor adaptation — analysis

Thin driver over the `tvns_sensorimotor` package. All logic lives in the package;
this notebook only orchestrates and displays. The messy exploratory notebook it
replaces is preserved as `../tVNS_sensorimotor_analysis copy.ipynb`.

**Checklist fixes are baked into the code** — see `CHECKLIST_RESPONSE.md` for the
item-by-item map. Highlights:

| item | where | effect |
|---|---|---|
| B1 sign convention | `prep.add_baseline_correction` | asserts improvement = baseline−run > 0 |
| B3 β labels | `stats_behavior.COEF_LABELS` | β₂ = mean level, γ₃ = slope diff |
| S1 Bonferroni | `fit_block_models` / `compare_block_slopes` | `p_bonf` across 8 blocks |
| S2 full tables | `fit_block_models` | β, SE, 95% CI, stat, p, p_bonf, n, RE var, OLS-fallback flag |
| B9 slope stats | `compare_block_slopes` | t, df, p, p_bonf, Cohen's d, mean±SD |
| B10 circularity | `independent_subject_slopes` | per-subject OLS, no treatment term |
| — model selection | `fit_overall_models` | AIC/BIC across 9 RE/FE specs (3×3 grid), pooled over blocks |
| — cluster permutation | `cluster_permutation.run_cluster_tests` | Stim vs Sham, RMSE_Needle / RMSE_Airspeed, 1-D cluster-mass test over trials |
| S11 retention | `retention` module | DR-day raw / % change / trajectory (was: none run) |
| B4 pupil direction | `stats_physio.interpret_slopes` | prints "shallower / steeper / opposite" |
| B6 physio model | `config.PHYSIO_RE_FORMULA` | random intercept **and** slope, stated once |
| S4 log-RMSSD | `fit_physio_models` | adds `log(RMSSD)` + no-truncation sensitivity |
| S12 missingness | `prep.missingness_table` | n present / missing per measure × group |
| B5 EEG gamma | `config.EEG_BANDS` | gamma = 30–55 Hz for the 1–60 Hz filter |
| B7 EEG permutation | `eeg.summarize_clusters` | emits the honest "no cluster survived" sentence |
| M1 EEG params | `eeg.describe_preprocessing` | Welch / permutation params surfaced |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd; pd.set_option('display.width', 160, 'display.max_columns', 40)

from tvns_sensorimotor import config, prep, stats_behavior as sb, stats_physio as sp
from tvns_sensorimotor import eeg, arousal, retention, cluster_permutation as cp, figures

# Point these at your local copies if they are not beside the notebook:
# config.DATA = config.DataPaths(physio='/path/final_physio_table.csv',
#                                eeg_features='/path/eeg_feats.csv',
#                                eeg_raw='/path/nn_data.pkl')
config.DATA.performance

## 1. Behaviour — data prep, baseline equivalence

In [ ]:
trials = prep.build_trials_df()   # load → block/trial labels → needle composite → baseline-correct (B1 checked)
print(trials.shape, trials.subject.nunique(), 'subjects')
sb.baseline_equivalence(trials)   # S3: groups equivalent at baseline in level and slope?

## 2. Table 1 — per-block mixed models
`improve ~ trial_c * treatment + (1 + trial_c | subject)`, OLS fallback flagged.
`p_bonf` is Bonferroni across the 8 blocks **per coefficient** (S1).

In [ ]:
table1 = sb.fit_block_models(trials, 'RMSE_Needle')
cols = ['block','coef','beta','se','ci_low','ci_high','stat','p','p_bonf','sig_bonf','ols_fallback']
table1[table1.term != 'Intercept'][cols]

## 3. Table 3 — within-block learning-slope comparison (B10)
Slopes are one **independent OLS fit per subject×block** with no treatment term,
then Welch t-test Stim vs Sham, Bonferroni across blocks. Contrast with the
manuscript's BLUP-based test.

In [ ]:
slopes = sb.independent_subject_slopes(trials, 'RMSE_Needle')
sb.compare_block_slopes(slopes)

In [ ]:
figures.figure1_learning_curves(trials);
figures.figure1_pct_learning_curves(trials);
figures.figure2_block_learning(trials, table1);

## 4. Cluster-based permutation test (Stim vs Sham)
1-D independent-samples cluster-permutation test over the full training trial axis (all 48 post-baseline trials), on `RMSE_Needle_improve` / `RMSE_Airspeed_improve` — the same baseline-corrected quantity Figure 1 plots. Unlike Table 1 (per-block) or the model-selection ladder below (a single linear covariate), this makes no assumption about *where* in training a group difference would show up: it finds any contiguous run of trials where Stim and Sham differ, with multiplicity handled by the cluster-mass permutation null rather than a per-trial Bonferroni (which would have essentially no power at N=28/arm across ~48 trials).

Same method as `stats_physio.trial_cluster_test` (Table 2 / Figure 3) and `eeg.run_cluster_tests` (Figure 5) — shared implementation in `_common.cluster_test_over_index`. An empty result is reported as exactly that, never explained away (cf. B7).

In [ ]:
clusters = cp.run_cluster_tests(trials)
print(cp.summarize_clusters(clusters))
cp.figure_cluster_overlay(trials, clusters);

Raw, uncorrected per-trial p-values (Welch t-test, Stim vs Sham) underlying the cluster test above, for both `RMSE_Needle` and `RMSE_Airspeed` and both the raw improvement (`baseline − RMSE`) and percent-change (`(baseline − RMSE) / baseline × 100`) versions. Shown for inspection only — at 48 trials, treating any single `p < .05` here as a finding repeats exactly the multiple-comparisons problem the cluster test above (and S1's Bonferroni) exists to avoid.

In [ ]:
pointwise = cp.pointwise_ttests(trials)
pointwise[pointwise.p < 0.05].sort_values(['measure', 'value_kind', 'p'])

## 5. Overall model comparison — random-effects structure (AIC/BIC)
Pooled over all 8 blocks (Block entered as a covariate rather than stratified
on), nine specs on `RMSE_Needle_improve` / `RMSE_Airspeed_improve`. Trial =
trial-within-block (1–6, centred); Block = block number (1–8), centred and
continuous (a single linear covariate, matching how Trial is a single
coefficient rather than a per-block dummy). Three fixed-effects structures ×
three random-effects structures — a 3×3 grid:

| | fixed effects | intercept only | + slope on Trial | + slope on Block |
|---|---|---|---|---|
| no interaction | `Treatment+Trial+Block` | M3 | M7 | M10 |
| Treatment:Trial | `+Treatment:Trial` | M2 | M1 | M8 |
| Treatment:Block | `+Treatment:Block` | M5 | M11 | M4 |

M1 vs M2 / M3 vs M7 ask whether a random Trial slope is warranted, with vs
without the Treatment:Trial fixed interaction. M4 vs M5 / M10 vs M3 ask the
same for a random Block slope, with vs without a fixed interaction present at
all. M1/M2 vs M8 and M5 vs M11 each swap which random slope (Trial vs Block)
goes with a given fixed interaction. M3 vs M5 asks whether Treatment
moderates the across-block trend (Treatment:Block) rather than the
within-block trend (Treatment:Trial).

All nine are fit by **ML** (not REML, needed once the fixed-effect
structure changes) with the **same optimizer — powell** — for every spec, so
the AIC/BIC comparison isn't confounded by different specs landing on
different optimizers.

In [ ]:
overall_table, overall_fits = sb.fit_overall_models(trials, 'RMSE_Needle')
display(overall_table.drop(columns=['description', 'formula']))

best_aic = overall_table.loc[overall_table['aic'].idxmin(), 'model']
best_bic = overall_table.loc[overall_table['bic'].idxmin(), 'model']
print(f"best by AIC: {best_aic}" + ("" if best_aic == best_bic else f"  |  best by BIC: {best_bic}"))
sb.tidy_mixedlm_coefficients(overall_fits[best_aic], best_aic)

## 6. Post-training retention (DR days) — Stim vs Sham (S11)
Needle and Airspeed RMSE, three views: (a) raw scores at each **retention**
day, Welch t-test + bar chart of mean ± SEM; (b) baseline-normalized %
**improvement** at each retention day (`(baseline − day) / baseline * 100` —
positive = better than baseline, same sign convention as `_improve`
elsewhere), same test + bar chart; (c) the % improvement **trajectory across
the whole study**, D00 (0 % by construction) → every training day (D01..D04)
→ every retention day. Only needs the committed performance CSV —
`prep.build_retention_df` keeps every day (D00 is excluded only from the main
training pipeline's block/trial analysis, `prep.build_trials_df`, not from
here); the raw/%-improvement *t-tests* still restrict to the DR days
specifically, since those are the actual retention comparisons.

In [ ]:
retention_df = prep.build_retention_df()
raw_tt = retention.retention_raw_ttests(retention_df)
pct_tt = retention.retention_pct_change_ttests(retention_df)
display(raw_tt)
display(pct_tt)

retention.figure_retention_raw(retention_df, raw_tt);
retention.figure_retention_pct_change(retention_df, pct_tt);
retention.figure_retention_trajectory(retention_df);

## 7. Table 2 — peripheral physiology  *(needs `final_physio_table.csv`)*
One model form (B6): `measure ~ trial_c * treatment + (1 + trial_c | subject)`.
Adds `log(RMSSD)` and a no->200 ms-truncation sensitivity fit (S4).

In [ ]:
physio = prep.build_physio_df(trials)
table2 = sp.fit_physio_models(physio)
display(table2[['model','coef','beta','se','ci_low','ci_high','z','p','p_bonf']])
display(sp.interpret_slopes(table2))          # B4: direction in words vs a-priori
display(prep.missingness_table(physio, config.PHYSIO_MEASURES))   # S12
figures.figure3_physio(physio, table2,
    clusters={m: sp.trial_cluster_test(physio, m) for m in config.PHYSIO_MEASURES});

## 8. Figure 4 — arousal latent variable  *(needs `eeg_feats.csv` + physio)*

In [ ]:
adf = prep.build_arousal_df(physio)
scores, loadings, evr = arousal.fit_pca(adf)
print('explained variance', evr.round(3)); display(loadings.round(3))
display(arousal.pc_learning_models(arousal.add_phase(scores)))
figures.figure4_arousal(scores, loadings);

## 9. Figure 5 — EEG band power + cluster permutation  *(needs `nn_data.pkl`)*
B5: gamma is 30–55 Hz. B7: report exactly what survives — usually nothing.

In [ ]:
print(eeg.describe_preprocessing())
band_power, raw = eeg.compute_band_power()
results = eeg.run_cluster_tests(band_power, raw)
display(pd.DataFrame([{k: v for k, v in vars(r).items() if k not in ('diff_topo','sig_mask')} for r in results]))
print(eeg.summarize_clusters(results))

## 10. Regenerate everything
```python
from tvns_sensorimotor import run_all
res = run_all.run(save=True)          # behaviour + retention
res = run_all.run(save=True, do_physio=True, do_eeg=True, do_arousal=True)
```
or from a shell: `python -m tvns_sensorimotor.run_all --all`
(retention runs by default; add `--no-retention` to skip it).